<a href="https://colab.research.google.com/github/mhu55ien/ml-from-scratch/blob/main/notebooks/01_linear_regression_gradient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import sklearn
print(np.__version__, sklearn.__version__)

2.4.6 1.6.1


In [2]:
def Compute_cost(X, y, w ,b):
  m,n = X.shape
  f = np.dot(X,w)+b
  cost = np.sum((f-y)**2)/(2*m)
  return cost

In [3]:
def Compute_gradient(X, y, w, b):
  m,n = X.shape
  f = np.dot(X,w)+b
  e = (f-y)


  gradient_w = np.dot(e,X)/(m)
  gradient_b = np.sum(e)/m
  gradient = (gradient_w, gradient_b)
  return gradient





In [4]:
X = np.array([[1.], [2.]])
y = np.array([2., 3.])
w = np.array([0.])
b = 0.

assert np.isclose(Compute_cost(X, y, w, b), 3.25)
gw, gb = Compute_gradient(X, y, w, b)
assert np.allclose(gw, [-4.])
assert np.isclose(gb, -2.5)
print("all good")

all good


In [5]:
rng = np.random.default_rng(0)

# ---------- reference implementations (used only by the tests) ----------
def gradient_loops(X, y, w, b):
    """The formula transcribed literally: explicit sums over i, one slot per j."""
    m, n = X.shape
    gw = np.zeros(n)
    gb = 0.0
    for i in range(m):
        err = np.dot(X[i], w) + b - y[i]
        for j in range(n):
            gw[j] += err * X[i, j]
        gb += err
    return gw / m, gb / m

def numerical_gradient(X, y, w, b, eps=1e-5):
    """Central finite differences on the cost: slope measured without any calculus."""
    gw = np.zeros_like(w)
    for j in range(len(w)):
        w_plus, w_minus = w.copy(), w.copy()
        w_plus[j] += eps
        w_minus[j] -= eps
        gw[j] = (Compute_cost(X, y, w_plus, b) - Compute_cost(X, y, w_minus, b)) / (2 * eps)
    gb = (Compute_cost(X, y, w, b + eps) - Compute_cost(X, y, w, b - eps)) / (2 * eps)
    return gw, gb

# ---------- Test 1: hand-computed example ----------
X = np.array([[1.], [2.]])
y = np.array([2., 3.])
w = np.array([0.])
b = 0.
assert np.isclose(Compute_cost(X, y, w, b), 3.25)
gw, gb = Compute_gradient(X, y, w, b)
assert np.allclose(gw, [-4.])
assert np.isclose(gb, -2.5)
assert gw.shape == w.shape

# ---------- Test 2: shapes with several features ----------
X3 = rng.standard_normal((5, 3))
y3 = rng.standard_normal(5)
w3 = rng.standard_normal(3)
b3 = 0.5
gw3, gb3 = Compute_gradient(X3, y3, w3, b3)
assert gw3.shape == (3,)
assert np.ndim(gb3) == 0          # b's gradient is a scalar

# ---------- Test 3: vectorized == literal formula ----------
gw_ref, gb_ref = gradient_loops(X3, y3, w3, b3)
assert np.allclose(gw3, gw_ref)
assert np.isclose(gb3, gb_ref)

# ---------- Test 4: analytic == numerical (gradient check) ----------
gw_num, gb_num = numerical_gradient(X3, y3, w3, b3)
rel_w = np.linalg.norm(gw3 - gw_num) / (np.linalg.norm(gw3) + np.linalg.norm(gw_num))
assert rel_w < 1e-7, rel_w
assert abs(gb3 - gb_num) < 1e-7

# ---------- Test 5: at the true parameters, cost and gradient are zero ----------
Xp = rng.standard_normal((20, 3))
w_true = np.array([2., -1., 0.5])
b_true = 3.
yp = Xp @ w_true + b_true          # data generated with no noise
assert np.isclose(Compute_cost(Xp, yp, w_true, b_true), 0.)
gwp, gbp = Compute_gradient(Xp, yp, w_true, b_true)
assert np.allclose(gwp, 0., atol=1e-12)
assert np.isclose(gbp, 0., atol=1e-12)

# ---------- Test 6: same cost, opposite gradient ----------
y_over = np.array([-2., -3.])      # predictions now overshoot instead of undershoot
assert np.isclose(Compute_cost(X, y_over, w, b), Compute_cost(X, y, w, b))
gw_over, gb_over = Compute_gradient(X, y_over, w, b)
assert np.allclose(gw_over, -gw)
assert np.isclose(gb_over, -gb)

print("all tests passed")

all tests passed
